# This model is trained on (2020/01/01  to  2023/12/30) 48 month data

In [8]:
import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

# If it is available, check how many GPUs are detected
if cuda_available:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")

# Check the CUDA version PyTorch was built with
print(f"PyTorch CUDA Version: {torch.version.cuda}")

Is CUDA available? True
Number of GPUs: 2
Current GPU Name: NVIDIA GeForce GTX 1080 Ti
PyTorch CUDA Version: 12.1


In [11]:
# ==============================================================================
# MASTER CONFIGURATION CELL
# ==============================================================================
import torch
import os

# --- Core Parameters ---
# MODIFICATION: Set a range or a specific value for the lookback period
LOOKBACK_HOURS = 48  # Change this to 24, 36, etc., to test different models
# MODIFICATION: Increased forecast period to 7 days (7 * 24 = 168)
FORECAST_HORIZON_HOURS = 168

# --- File Paths ---
# MODIFICATION: Updated to the new raw data path you provided
RAW_DATA_PATH = r'c:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'

# MODIFICATION: File paths are now dynamic based on the lookback period
# This prevents experiments from overwriting each other.
BASE_DIR = r'D:\babe_prediction'
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, f'processed_data_lookback_{LOOKBACK_HOURS}')
MODEL_SAVE_PATH = f'convlstm_lookback_{LOOKBACK_HOURS}_forecast_{FORECAST_HORIZON_HOURS}.pth'

# --- Feature Engineering ---
VARS_TO_USE = ['VCMX','VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
TARGET_VAR = 'VCMX'

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-5
BATCH_SIZE = 8
EPOCHS = 50
# MODIFICATION: Early stopping patience is now a configurable parameter
EARLY_STOPPING_PATIENCE = 5

# --- System Configuration ---
NUM_WORKERS = 0  # Set to 0 for Windows compatibility, 4 or more for Linux
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print a summary of the current configuration
print("--- Configuration Summary ---")
print(f"Lookback Period: {LOOKBACK_HOURS} hours")
print(f"Forecast Horizon: {FORECAST_HORIZON_HOURS} hours")
print(f"Processed Data Path: {PROCESSED_DATA_DIR}")
print(f"Model Save Path: {MODEL_SAVE_PATH}")
print(f"Using Device: {device}")
print("---------------------------")

--- Configuration Summary ---
Lookback Period: 48 hours
Forecast Horizon: 168 hours
Processed Data Path: D:\babe_prediction\processed_data_lookback_48
Model Save Path: convlstm_lookback_48_forecast_168.pth
Using Device: cuda
---------------------------


# STEP 1: PRE-PROCESSING SCRIPT

In [ ]:
# ==============================================================================
# ==============================================================================
import os
import pickle
import xarray as xr
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import shutil

warnings.filterwarnings('ignore')

# --- Clean up old processed data if it exists ---
if os.path.exists(PROCESSED_DATA_DIR):
    print(f"Removing old processed data directory: {PROCESSED_DATA_DIR}")
    shutil.rmtree(PROCESSED_DATA_DIR)

print("\n--- Starting Full Pre-processing Workflow ---")

# ==============================================================================
# Step 1: Load and Clean Raw Data
# ==============================================================================
print("\n[Step 1/4] Loading and cleaning raw data...")
try:
    ds_raw = xr.open_dataset(RAW_DATA_PATH)
except FileNotFoundError:
    raise SystemExit(f"❌ ERROR: Raw data file not found at {RAW_DATA_PATH}")

ds_raw['VSDmag'] = np.sqrt(ds_raw['VSDX']**2 + ds_raw['VSDY']**2) 

ds_clean = ds_raw[VARS_TO_USE].astype(np.float32).fillna(0)
print("✅ Raw data loaded and NaN values filled with 0.")

# ==============================================================================
# Step 2: Define Data Splits
# ==============================================================================
print("\n[Step 2/4] Splitting data into train, validation, and test sets...")

# Note: Adjust these dates based on the new dataset's time range
ds_train = ds_clean.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds_clean.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds_clean.sel(time=slice('2023-07-01', '2023-12-30'))

ds_splits = {'train': ds_train, 'val': ds_val, 'test': ds_test}
print(f"Train split: {len(ds_train.time)} time steps")
print(f"Validation split: {len(ds_val.time)} time steps")
print(f"Test split: {len(ds_test.time)} time steps")

# ==============================================================================
# Step 3: Create and Save Scalers
# ==============================================================================
print("\n[Step 3/4] Fitting scalers on TRAINING data only...")

scalers = {}
for var in tqdm(VARS_TO_USE, desc="Fitting Scalers"):
    data_to_fit = ds_train[var].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    scaler.fit(data_to_fit)
    scalers[var] = scaler

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
scaler_path = os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scalers, f)
print(f"✅ Scalers fitted and saved to '{scaler_path}'")

# ==============================================================================
# Step 4: Process and Save Samples using a Sliding Window
# ==============================================================================
print("\n[Step 4/4] Generating and saving individual samples...")

total_window_size = LOOKBACK_HOURS + FORECAST_HORIZON_HOURS

for split_name, ds_split in ds_splits.items():
    print(f"\n--- Processing '{split_name}' split ---")
    split_dir = os.path.join(PROCESSED_DATA_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    num_sequences = len(ds_split['time']) - total_window_size
    if num_sequences < 0:
        print(f"⚠️ Warning: '{split_name}' split is too small to create any sequences. Skipping.")
        continue
        
    for i in tqdm(range(num_sequences), desc=f"Saving {split_name} samples"):
        window_slice = ds_split.isel(time=slice(i, i + total_window_size))
        
        scaled_window_vars = []
        for var in VARS_TO_USE:
            data = window_slice[var].values
            scaled_data = scalers[var].transform(data.reshape(-1, 1)).reshape(data.shape)
            scaled_window_vars.append(scaled_data)
        
        scaled_window_tensor = np.stack(scaled_window_vars, axis=1)

        X_np = scaled_window_tensor[:LOOKBACK_HOURS, :, :, :]
        y_np = scaled_window_tensor[LOOKBACK_HOURS:, VARS_TO_USE.index(TARGET_VAR), :, :]

        X = torch.from_numpy(X_np.astype(np.float32))
        y = torch.from_numpy(y_np.astype(np.float32))
        
        sample_path = os.path.join(split_dir, f'sample_{i:06d}.pt')
        torch.save((X, y), sample_path)

print("\n\n✅ Pre-processing complete!")
print(f"Clean, processed data is now available at: '{PROCESSED_DATA_DIR}'")

Removing old processed data directory: D:\babe_prediction\processed_data_lookback_48

--- Starting Full Pre-processing Workflow ---

[Step 1/4] Loading and cleaning raw data...


# STEP 2: TRAINING SCRIPT

In [12]:
# ==============================================================================
# STEP 2: TRAINING SCRIPT (MODIFIED FOR SPEED)
# ==============================================================================
import os
import pickle
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time
import numpy as np

# --- You can leave all the class definitions as they are ---
class PreprocessedWaveDataset(Dataset):
    def __init__(self, split_dir):
        self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim, self.hidden_dim, self.kernel_size, self.bias = input_dim, hidden_dim, kernel_size, bias
        self.padding = kernel_size[0] // 2
        self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)

    def forward(self, x, h_c):
        h, c = h_c
        combined = torch.cat([x, h], dim=1)
        cc = self.conv(combined)
        i, f, o, g = torch.split(cc, self.hidden_dim, dim=1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_n = f * c + i * g
        h_n = o * torch.tanh(c_n)
        return h_n, c_n

    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=device))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__()
        self.batch_first, self.num_layers = batch_first, num_layers
        hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim
        cell_list = []
        for i in range(self.num_layers):
            cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]
            cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias))
        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, x, h_c=None):
        b, s_l, _, h, w = x.size()
        if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
        
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]
            output_inner = []
            for t in range(s_l):
                h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c])
                output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
            
        return cur_in, [h, c]
        
    def _init_hidden(self, batch_size, image_size, device):
        return [cell.init_hidden(batch_size, image_size, device) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__()
        self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True)
        self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True)
        self.output_conv = nn.Conv2d(hidden_dims[1], forecast_horizon, kernel_size=(1, 1), padding='same')
    
    def forward(self, x_seq):
        l1_o, _ = self.cl1(x_seq)
        l2_o, _ = self.cl2(l1_o)
        last_time_step_features = l2_o[:, -1, :, :, :]
        return self.output_conv(last_time_step_features)

def train_model(model, train_loader, val_loader, device, epochs, lr, patience, model_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val_loss = float('inf')
    scaler = torch.cuda.amp.GradScaler()
    epochs_no_improve = 0
    
    print("\n--- Starting Model Training ---")
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        for X, y in train_pbar:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                predicted = model(X)
                target = y
                loss = loss_fn(predicted, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_train_loss += loss.item()
            train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})

        avg_train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                with torch.cuda.amp.autocast():
                    predicted = model(X)
                    target = y
                    loss = loss_fn(predicted, target)
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_path)
            print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
                
    print(f"\n✅ Training completed. Best model saved to {model_path}")
    model.load_state_dict(torch.load(model_path))
    return model


In [13]:
# MODIFICATION: Wrap the execution logic in this block for stable multiprocessing
if __name__ == '__main__':
    # --- Main Execution Logic ---
    # Using parameters from the Configuration Cell
    train_dir = os.path.join(PROCESSED_DATA_DIR, 'train')
    val_dir = os.path.join(PROCESSED_DATA_DIR, 'val')

    train_dataset = PreprocessedWaveDataset(train_dir)
    val_dataset = PreprocessedWaveDataset(val_dir)

    # MODIFICATION: Added persistent_workers=True to the DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS, 
        pin_memory=True,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS, 
        pin_memory=True,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )

    print(f"\nDataLoaders created with {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

    INPUT_CHANNELS = len(VARS_TO_USE)
    model = ConvLSTMNet(input_dim=INPUT_CHANNELS, forecast_horizon=FORECAST_HORIZON_HOURS)
    model.to(device)
    print(f"Model built with {INPUT_CHANNELS} input channels and a {FORECAST_HORIZON_HOURS}-hour forecast horizon.")

    trained_model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        patience=EARLY_STOPPING_PATIENCE,
        model_path=MODEL_SAVE_PATH
    )


DataLoaders created with 157 training samples and 0 validation samples.
Model built with 9 input channels and a 168-hour forecast horizon.

--- Starting Model Training ---


Epoch 1/50 [Training]:   0%|          | 0/20 [00:00<?, ?it/s]

ZeroDivisionError: float division by zero

# STEP 3: EVALUATION AND VISUALIZATION SCRIPT

In [ ]:
# ==============================================================================
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import pickle
import glob
from torch.utils.data import Dataset, DataLoader

print("--- Starting Final Evaluation on the Test Set ---\n")

# --- Define All Necessary Classes ---
# (Pasting classes again for a self-contained script)
class PreprocessedWaveDataset(Dataset):
    def __init__(self, split_dir):
        self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim, self.hidden_dim, self.kernel_size, self.bias = input_dim, hidden_dim, kernel_size, bias
        self.padding = kernel_size[0] // 2
        self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
    def forward(self, x, h_c): h, c = h_c; combined = torch.cat([x, h], dim=1); cc = self.conv(combined); i, f, o, g = torch.split(cc, self.hidden_dim, dim=1); i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g); c_n = f * c + i * g; h_n = o * torch.tanh(c_n); return h_n, c_n
    def init_hidden(self, b, i, d): h, w = i; return (torch.zeros(b, self.hidden_dim, h, w, device=d), torch.zeros(b, self.hidden_dim, h, w, device=d))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__(); self.batch_first, self.num_layers = batch_first, num_layers; hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim; cell_list = [];
        for i in range(self.num_layers): cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]; cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias)); self.cell_list = nn.ModuleList(cell_list)
    def forward(self, x, h_c=None):
        b, s_l, _, h, w = x.size();
        if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]; output_inner = []
            for t in range(s_l): h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c]); output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
        return cur_in, [h, c]
    def _init_hidden(self, b, i, d): return [cell.init_hidden(b, i, d) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__(); self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True); self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True); self.output_conv = nn.Conv2d(hidden_dims[1], forecast_horizon, kernel_size=(1, 1), padding='same')
    def forward(self, x_seq): l1_o, _ = self.cl1(x_seq); l2_o, _ = self.cl2(l1_o); return self.output_conv(l2_o[:, -1, :, :, :])

# --- Load Model and Prepare Test Loader ---
try:
    with open(os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl'), 'rb') as f:
        scalers = pickle.load(f)
except FileNotFoundError:
    print(f"❌ ERROR: Scalers file not found. Make sure you re-ran the preprocessing script for lookback={LOOKBACK_HOURS}.")
    exit()

test_dir = os.path.join(PROCESSED_DATA_DIR, 'test')
test_dataset = PreprocessedWaveDataset(test_dir)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = ConvLSTMNet(input_dim=INPUT_CHANNELS, forecast_horizon=FORECAST_HORIZON_HOURS)
try:
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
except FileNotFoundError:
    print(f"❌ ERROR: Model file '{MODEL_SAVE_PATH}' not found. Make sure you have trained the new model.")
    exit()

if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
model.to(device)
model.eval()
print("✅ Best model loaded successfully.")

# --- Evaluation and Visualization Functions (MODIFIED) ---
def evaluate_and_visualize(model, test_loader, device, scalers, target_var):
    model.eval()
    total_rmse_sum_sq, total_mae_sum = 0, 0
    total_samples = 0
    vis_data = {'predictions': [], 'targets': []}
    
    print("\n--- Generating predictions for the test set... ---\n")
    with torch.no_grad():
        for i, (X, y) in enumerate(tqdm(test_loader, desc="Evaluating Test Set")):
            X, y = X.to(device), y.to(device)
            predictions_scaled = model(X)
            
            target_scaled = y
            target_scaler = scalers[target_var]
            
            pred_original = target_scaler.inverse_transform(predictions_scaled.cpu().numpy().reshape(-1, 1)).reshape(predictions_scaled.shape)
            target_original = target_scaler.inverse_transform(target_scaled.cpu().numpy().reshape(-1, 1)).reshape(target_scaled.shape)
            
            total_rmse_sum_sq += np.sum((pred_original - target_original) ** 2)
            total_mae_sum += np.sum(np.abs(pred_original - target_original))
            total_samples += pred_original.size
            
            if len(vis_data['predictions']) < 2: # Get 2 samples for plotting
                vis_data['predictions'].append(pred_original[0])
                vis_data['targets'].append(target_original[0])

    rmse = np.sqrt(total_rmse_sum_sq / total_samples)
    mae = total_mae_sum / total_samples
    print(f"\n📊 Test Results (across all {FORECAST_HORIZON_HOURS} hours):\n  RMSE: {rmse:.4f}\n  MAE:  {mae:.4f}")
    
    return vis_data

def plot_visual_comparison(vis_data, save_path, timesteps_to_plot):
    num_timesteps = len(timesteps_to_plot)
    for i in range(len(vis_data['predictions'])):
        fig, axes = plt.subplots(num_timesteps, 3, figsize=(18, 5 * num_timesteps), squeeze=False)
        plt.suptitle(f"Visual Comparison for Sample {i+1}", fontsize=18, y=0.99)
        
        for row, t_idx in enumerate(timesteps_to_plot):
            pred_sample = vis_data['predictions'][i][t_idx, :, :]
            targ_sample = vis_data['targets'][i][t_idx, :, :]
            v_max = max(np.max(pred_sample), np.max(targ_sample), 0.1)
            
            im1 = axes[row, 0].imshow(targ_sample, cmap='viridis', vmin=0, vmax=v_max)
            axes[row, 0].set_title(f'Target (Hour {t_idx+1})'); fig.colorbar(im1, ax=axes[row, 0])
            
            im2 = axes[row, 1].imshow(pred_sample, cmap='viridis', vmin=0, vmax=v_max)
            axes[row, 1].set_title(f'Prediction (Hour {t_idx+1})'); fig.colorbar(im2, ax=axes[row, 1])
            
            diff = pred_sample - targ_sample
            diff_max = np.max(np.abs(diff)) if np.max(np.abs(diff)) > 0 else 0.1
            im3 = axes[row, 2].imshow(diff, cmap='RdBu_r', vmin=-diff_max, vmax=diff_max)
            axes[row, 2].set_title(f'Error (Hour {t_idx+1})'); fig.colorbar(im3, ax=axes[row, 2])
            
        plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.savefig(f"{save_path}_{i}.png", dpi=200); plt.show()
    print(f"✅ Comparison maps saved to '{save_path}_X.png'")

# --- Run Evaluation ---
vis_data = evaluate_and_visualize(model, test_loader, device, scalers, TARGET_VAR)

# --- Generate Visualizations ---
print("\n--- Generating Visualization 1: Comparison Maps ---")
plot_visual_comparison(
    vis_data, 
    save_path=f'test_comparison_lookback_{LOOKBACK_HOURS}',
    # MODIFICATION: Plotting Hour 1, Day 1, Day 3, and Day 7
    timesteps_to_plot=[0, 23, 71, 167] 
)

print("\n--- Generating Visualization 2 & 3: Scatter and Error Plots ---")
all_preds_flat = np.array(vis_data['predictions']).flatten()
all_targets_flat = np.array(vis_data['targets']).flatten()
errors = all_preds_flat - all_targets_flat

# Scatter Plot
plt.figure(figsize=(8, 8))
sample_indices = np.random.choice(len(all_preds_flat), min(len(all_preds_flat), 10000), replace=False)
plt.scatter(all_targets_flat[sample_indices], all_preds_flat[sample_indices], alpha=0.3, s=10)
min_val = min(all_targets_flat.min(), all_preds_flat.min())
max_val = max(all_targets_flat.max(), all_preds_flat.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel("Actual Wave Height (m)"); plt.ylabel("Predicted Wave Height (m)")
plt.title(f"Scatter Plot (Lookback: {LOOKBACK_HOURS}hrs, Forecast: {FORECAST_HORIZON_HOURS}hrs)"); plt.grid(True); plt.legend(); plt.axis('equal'); plt.tight_layout()
plt.savefig(f'scatter_plot_lookback_{LOOKBACK_HOURS}.png', dpi=300); plt.show()
print(f"✅ Scatter plot saved to 'scatter_plot_lookback_{LOOKBACK_HOURS}.png'")

# Error Histogram
mean_error = np.mean(errors)
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=100, density=True)
plt.axvline(mean_error, color='r', linestyle='--', lw=2, label=f'Mean Error: {mean_error:.3f}')
plt.title(f"Distribution of Prediction Errors (Lookback: {LOOKBACK_HOURS}hrs)")
plt.xlabel("Error (Predicted - Actual) in meters"); plt.ylabel("Density")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(f'error_histogram_lookback_{LOOKBACK_HOURS}.png', dpi=300); plt.show()
print(f"✅ Error histogram saved to 'error_histogram_lookback_{LOOKBACK_HOURS}.png'")

print("\n✅ All evaluation visualizations are complete.")